# 03b Join trigger events with a stimulus log

This notebook is optional and generic. It joins the trigger events of each recording with a stimulus log, row by row in presentation order, and writes the result as analysis events:

    derivatives/meeg-pipeline/sub-*/meg/events/*_desc-stimulus_events.tsv

Use it when the trigger channel only marks that a stimulus started and the stimulus properties (frequency, level, segment, ...) are in a separate file. It does not overwrite the raw BIDS `*_events.tsv`.

Nothing is repaired silently: a different number of rows or, with `match: "code"`, a trigger value that differs from the log raises a `mismatch` status for that recording.

Settings live in `configs/local.yaml` under `events.stimulus_log`. The same step is available from the terminal: `meegpipe events-join-stimulus-log --config configs/local.yaml --subject <id>`.

The quality check fits `trigger onset = intercept + slope * log onset`. The slope is the clock ratio between the stimulus computer and the MEG system; the residuals show jitter and drift.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.event_derivatives import (
    join_stimulus_log_for_recordings,
    stimulus_log_join_results_to_dataframe,
    stimulus_log_path,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

## Selection

`iter_recordings(..., "all")` uses existing raw-BIDS recordings and excludes `sub-emptyroom` by default.

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

RUN_RESIDUAL_PLOT = True

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)

## Overwrite policy

Default: skip existing joined events. Set `OVERWRITE_STEPS = ["analysis_events"]` to recompute them.

In [ ]:
OVERWRITE_STEPS = []

analysis_events_policy = existing_output_policy_for_step(
    "analysis_events",
    OVERWRITE_STEPS,
)

pd.DataFrame(
    [
        {
            "step": "analysis_events",
            "overwrite": should_overwrite("analysis_events", OVERWRITE_STEPS),
            "policy": analysis_events_policy,
        }
    ]
)

## Settings

In [ ]:
STIMULUS_LOG = config.events.stimulus_log

pd.DataFrame([STIMULUS_LOG.__dict__]).T

## Stimulus-log files

Checks which of the selected recordings have a log file at the configured path.

In [ ]:
pd.DataFrame(
    [
        {
            "subject": recording["subject"],
            "session": recording["session"],
            "task": recording["task"],
            "run": recording["run"],
            "stimulus_log_path": stimulus_log_path(config, recording),
            "exists": (
                stimulus_log_path(config, recording) is not None
                and stimulus_log_path(config, recording).exists()
            ),
        }
        for recording in selected_recordings
    ]
)

## Join

Existing outputs are skipped unless `OVERWRITE_STEPS` contains `"analysis_events"`. Missing inputs and mismatches are returned as status rows rather than stopping the whole batch.

In [ ]:
join_results = join_stimulus_log_for_recordings(
    config,
    selected_recordings,
    on_existing=analysis_events_policy,
)

join_results_df = stimulus_log_join_results_to_dataframe(join_results)
join_results_df

## Result summary

In [ ]:
if join_results_df.empty:
    pd.DataFrame()
else:
    (
        join_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Clock ratio and timing residuals

`slope` is the clock ratio (1.0 means identical clocks), `residual_sd_s` the jitter of the trigger onsets around the fitted line.

In [ ]:
finished = join_results_df[
    join_results_df["status"].isin(["written", "skipped_existing"])
] if not join_results_df.empty else join_results_df

finished[
    [
        "subject",
        "session",
        "task",
        "run",
        "n_events",
        "slope",
        "residual_sd_s",
        "max_abs_residual_s",
    ]
] if not finished.empty else pd.DataFrame()

## Residual plot

Timing residuals (trigger onset minus the fitted line) over the recording. A trend means clock drift, a wide band means jitter.

In [ ]:
if RUN_RESIDUAL_PLOT and not finished.empty:
    for _, row in finished.iterrows():
        joined = pd.read_csv(row["events_path"], sep="\t")
        onset_log = (
            joined["onset_stim"]
            if "onset_stim" in joined.columns
            else joined[STIMULUS_LOG.onset_column]
        ).to_numpy(dtype=float)
        onset_trigger = joined["onset"].to_numpy(dtype=float)
        slope, intercept = np.polyfit(onset_log, onset_trigger, 1)
        residuals_ms = (onset_trigger - (intercept + slope * onset_log)) * 1000

        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(onset_trigger, residuals_ms, ".", markersize=3)
        ax.axhline(0, color="0.5", linewidth=0.8)
        ax.set(
            title=f"{row['subject']} {row['task'] or ''} {row['run'] or ''}".strip(),
            xlabel="Trigger onset (s)",
            ylabel="Residual (ms)",
        )
        plt.show()
else:
    print("Skipped residual plot.")

## Preview joined events

First rows of the joined table of the first finished recording.

In [ ]:
if finished.empty:
    pd.DataFrame()
else:
    pd.read_csv(finished["events_path"].iloc[0], sep="\t").head(30)